# 04 — Model Lift & Decile Analysis

## Purpose
Translate model performance into operational language that a retention team
can act on. AUC tells us the model *can* rank customers by risk; lift analysis
tells us *how efficiently* it does so when applied to a real outreach budget.

## Why Lift Matters
A retention team can rarely contact every customer. Lift analysis answers:

> "If I can only reach **X%** of my customer base, what fraction of all
> churners will I capture by following the model's ranked list?"

## Key Results
| Metric                                    | Value      |
|-------------------------------------------|------------|
| Top decile lift (highest-risk 10%)        | **1.84×**  |
| Recall at top 20% (% of churners caught)  | **35.2%**  |
| Baseline churn rate (random contact)      | ~22–25%    |

**Practical read:** Targeting the top 10% of customers by predicted risk means
your retention team is spending its calls on customers who churn at **84% above
the average rate**. Targeting the top 20% captures more than a third of all
future churners — 1.76× better than random outreach.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import xgboost as xgb

STEEL_BLUE_900 = "#003865"
STEEL_BLUE_500 = "#0072CE"

df = pd.read_csv("../data/telco_churn_225k_v120.csv")
X  = df.drop(columns=["customer_id", "churn", "churn_score", "billing_risk_score",
                       "cltv", "estimated_margin_pct"], errors="ignore")
y  = df["churn"]

for col in ["billing_call_resolved", "tech_issue_resolved"]:
    if col in X.columns:
        X[col] = X[col].fillna(-1).astype("int64")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
model = Pipeline(steps=[
    ("pre", ColumnTransformer(
        [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)],
        remainder="passthrough"
    )),
    ("clf", xgb.XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.08,
                               random_state=42, n_jobs=-1))
])
model.fit(X_train, y_train)
y_proba = model.predict_proba(X_test)[:, 1]
print("Model trained.")

## Decile Assignment

Customers are sorted by predicted churn probability (descending), then split
into 10 equal-sized bins. **Decile 1 = highest predicted risk; Decile 10 = lowest.**

Using rank-based `qcut` (rather than probability-based cut points) ensures
perfectly even bin sizes, which makes lift numbers directly comparable across deciles.

In [ ]:
results_df = pd.DataFrame({"actual": y_test.values, "prob": y_proba})
results_df = results_df.sort_values("prob", ascending=False).reset_index(drop=True)
results_df["decile"] = pd.qcut(results_df.index, q=10,
                                labels=np.arange(1, 11), duplicates="drop")

baseline_rate = results_df["actual"].mean()

decile_stats = (
    results_df.groupby("decile", observed=True)
    .agg(total_customers=("actual", "count"), actual_churners=("actual", "sum"))
    .sort_index().reset_index()
)
decile_stats["churn_rate"]   = decile_stats["actual_churners"] / decile_stats["total_customers"]
decile_stats["lift"]         = decile_stats["churn_rate"] / baseline_rate
decile_stats["cum_gain_pct"] = (decile_stats["actual_churners"].cumsum()
                                / results_df["actual"].sum() * 100)

print(decile_stats[["decile", "total_customers", "churn_rate", "lift", "cum_gain_pct"]]
      .to_string(index=False))

## Executive Summary

The numbers below are the key talking points for any business conversation about
model deployment and retention budget sizing.

In [ ]:
n_deciles           = decile_stats.shape[0]
total_churners      = results_df["actual"].sum()
top_decile_churners = decile_stats.iloc[0]["actual_churners"]
top_20_gain         = decile_stats.iloc[min(1, n_deciles - 1)]["cum_gain_pct"]

print("=" * 60)
print("EXECUTIVE SUMMARY")
print("=" * 60)
print(f"  Total churners in test set  : {int(total_churners):,}")
print(f"  Baseline churn rate         : {baseline_rate:.2%}")
print(f"  Top decile churn rate       : {decile_stats.iloc[0]['churn_rate']:.1%}")
print(f"  Top decile lift             : {decile_stats.iloc[0]['lift']:.2f}×")
print(f"  Recall at top 20%           : {top_20_gain:.1f}%")
print()
print(f"  'The top 10% of customers by predicted risk churn at")
print(f"   {decile_stats.iloc[0]['lift']:.2f}× the average rate.'")
print(f"  'Targeting the top 20% captures {top_20_gain:.1f}% of all churners.'")

## Visualizations

### Lift by Decile
The bar chart shows that lift declines monotonically from Decile 1 to Decile 10
(as expected). The dashed baseline at 1.0 marks average performance. Deciles 6–10
are below baseline — these are the customers the model identifies as low risk.

### Cumulative Gain
The gain curve plots how many churners are captured as we work down the ranked
list. The gap between the model curve and the random-guess diagonal represents
the efficiency gain from model-driven targeting.

In [ ]:
colors = [STEEL_BLUE_900, "#005696"] + [STEEL_BLUE_500] * max(0, n_deciles - 2)

plt.figure(figsize=(10, 5))
sns.barplot(data=decile_stats, x="decile", y="lift", palette=colors[:n_deciles])
plt.axhline(1, color=STEEL_BLUE_900, linestyle="--", alpha=0.6, label="Baseline (1×)")
plt.title("Model Lift by Decile: Prioritizing High-Risk Segments",
          fontsize=14, fontweight="bold")
plt.xlabel("Risk Decile (1 = Highest Risk)")
plt.ylabel("Lift (× Average Churn Rate)")
plt.legend()
plt.tight_layout()
plt.savefig("../reports/Model_Decile_Lift_Final.png", dpi=150)
plt.show()

In [ ]:
x_rand    = np.arange(1, n_deciles + 1)
y_rand    = (x_rand / n_deciles) * 100
x_20      = decile_stats.iloc[min(1, n_deciles - 1)]["decile"]

plt.figure(figsize=(10, 5))
plt.plot(decile_stats["decile"], decile_stats["cum_gain_pct"],
         marker="o", color=STEEL_BLUE_900, linewidth=2, label="Model")
plt.plot(x_rand, y_rand, linestyle="--", color="gray", alpha=0.7, label="Random Guess")
plt.annotate(f"Top 20% captures {top_20_gain:.1f}% of churn",
             xy=(x_20, top_20_gain),
             xytext=(min(4, n_deciles), top_20_gain - 10),
             arrowprops=dict(facecolor=STEEL_BLUE_900, shrink=0.05),
             fontsize=11, fontweight="bold")
plt.title("Cumulative Gain: Efficiency of Targeted Retention",
          fontsize=14, fontweight="bold")
plt.xlabel("Risk Decile (Cumulative)")
plt.ylabel("% of Total Churners Captured")
plt.xticks(np.arange(1, n_deciles + 1))
plt.legend()
plt.tight_layout()
plt.savefig("../reports/Model_Cumulative_Gain_Final.png", dpi=150)
plt.show()